# TT/MPS基礎 05 — 右直交化（Right Orthogonalization）

## 今回の位置づけ

前回までに、

$$
\text{TT-SVD}
\rightarrow
\text{gauge freedom}
\rightarrow
\text{left-to-right QR sweep}
\rightarrow
L_2^T L_2 = I_{r_2}
$$

まで確認しました。

今回はその鏡像として、**右端から左へ向かう右直交化**だけを扱います。

### 今回やること

1. 第3コア $G_3$ の right unfolding を理解する
2. $A_3^T$ に reduced QR を適用する
3. $Q^T$ を新しい右直交コアにする
4. 残った $R^T$ を第2コアの右ボンドへ吸収する
5. 右直交性と、第2–第3コアの局所収縮保存を自分で確認する

**mixed-canonical form や orthogonality center にはまだ進みません。**

今回も、理論と検証条件までは示しますが、実装部分は `TODO` として残します。

## 1. 第3コアの right unfolding

3階 TT の右端コアは

$$
G_3\in\mathbb{R}^{r_2\times n_3\times 1}
$$

です。

右端ボンドは $1$ なので、最後の軸を除けば

$$
A_3(\alpha_2,i_3)
=
G_3(\alpha_2,i_3,1)
$$

という行列として見られます。

したがって、

$$
\boxed{
A_3\in\mathbb{R}^{r_2\times n_3}
}
$$

です。

### 右直交性

左直交では「列」を直交化しましたが、右直交では「行」を直交化したいので、

$$
\boxed{
A_3A_3^T=I_{r_2}
}
$$

を目標にします。

成分で書けば、

$$
\sum_{i_3=1}^{n_3}
G_3(\alpha_2,i_3,1)
G_3(\beta_2,i_3,1)
=
\delta_{\alpha_2\beta_2}.
$$

つまり、第3コアの各行は、右側サイトに対応する正規直交状態になります。

In [1]:
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)


def reconstruct_tt3(G1: torch.Tensor, G2: torch.Tensor, G3: torch.Tensor) -> torch.Tensor:
    """3個のTTコアから3階テンソルを再構成する。"""
    return torch.einsum("aib,bjc,ckd->ijk", G1, G2, G3)


# 右直交化だけに集中するための小さい3階TT
n1, n2, n3 = 4, 3, 5
r1, r2 = 2, 3

G1 = torch.randn(1, n1, r1)
G2 = torch.randn(r1, n2, r2)
G3 = torch.randn(r2, n3, 1)

X_before = reconstruct_tt3(G1, G2, G3)

print("G1:", tuple(G1.shape))
print("G2:", tuple(G2.shape))
print("G3:", tuple(G3.shape))
print("X :", tuple(X_before.shape))
print("r2 <= n3:", r2 <= n3)


G1: (1, 4, 2)
G2: (2, 3, 3)
G3: (3, 5, 1)
X : (4, 3, 5)
r2 <= n3: True


## 2. 演習1 — $G_3$ を行列 $A_3$ として見る

### 目的

$$
G_3\in\mathbb{R}^{r_2\times n_3\times 1}
$$

から、

$$
A_3\in\mathbb{R}^{r_2\times n_3}
$$

を作ります。

ここではまだ QR 分解しません。

### TODO

- `G3` の最後のサイズ $1$ の軸をどう扱えばよいか考える
- `A3.shape == (r2, n3)` になることを確認する

### 考えること

左直交化では

$$
(r_{k-1}n_k)\times r_k
$$

という形にしました。

今回は右直交化なので、**なぜ $r_2$ が行側に残るのか**を意識してください。

In [2]:
# TODO 1:
# G3 から A3 を作ってください。
#
# どれでもいい
A3 = G3.reshape(r2, n3)   # 今書いてある方
A3 = G3[:, :, 0]          # インデックス指定で軸を消す
A3 = G3.squeeze(-1)       # 長さ1の軸を落とす
#
print("A3 shape:", A3.shape)

print("TODO: G3 を right unfolding して A3 を作る")


A3 shape: torch.Size([3, 5])
TODO: G3 を right unfolding して A3 を作る


## 3. なぜ $A_3^T$ に QR をかけるのか

目標は

$$
A_3A_3^T=I_{r_2}
$$

です。

つまり、$A_3$ の**行**を直交化したい。

しかし通常の QR 分解

$$
B=QR
$$

では、$Q$ の**列**が直交します。

そこで $A_3$ を転置して、

$$
A_3^T\in\mathbb{R}^{n_3\times r_2}
$$

に reduced QR を適用します。

$$
A_3^T = QR
$$

ここで、

$$
Q^TQ=I_{r_2}.
$$

両辺を転置すると、

$$
A_3=R^TQ^T.
$$

したがって、新しい右直交コアに

$$
\widetilde A_3=Q^T
$$

を使えば、

$$
\widetilde A_3\widetilde A_3^T
=
Q^TQ
=
I_{r_2}
$$

となります。

---

ただし $R^T$ を捨てると TT 全体が変わるため、第2コアへ吸収します。

$$
\widetilde G_2(\alpha_1,i_2,\beta_2)
=
\sum_{\alpha_2}
G_2(\alpha_1,i_2,\alpha_2)
R^T(\alpha_2,\beta_2).
$$

左直交化では $R$ を**右隣**へ送りました。

右直交化では $R^T$ を**左隣**へ送ります。

## 4. 演習2 — 第3コアを右直交化する

### TODO

1. `A3.T` に `torch.linalg.qr(..., mode="reduced")` を適用する
2. 得られた $Q$ と $R$ の shape を確認する
3. $Q^T$ を3階テンソルへ戻して、新しい第3コアを作る
4. $R^T$ を `G2` の**右ボンド**へ吸収する

期待する shape は、

$$
Q\in\mathbb{R}^{n_3\times r_2},
\qquad
R\in\mathbb{R}^{r_2\times r_2}
$$

$$
\widetilde G_3\in\mathbb{R}^{r_2\times n_3\times 1}
$$

$$
\widetilde G_2\in\mathbb{R}^{r_1\times n_2\times r_2}
$$

です。

### 実装前に考えること

$R^T$ は

$$
G_2(\alpha_1,i_2,\alpha_2)
$$

のどの添字に作用するでしょうか？

添字 $\alpha_2$ が、第2コアと第3コアをつないでいることを使って判断してください。

In [7]:
# TODO 2:
# A3.T の reduced QR と、R.T の G2 への吸収を実装してください。
#
# A3.T = Q3 @ R3  ⇒  A3 = R3.T @ Q3.T
# 新しい右直交コアは Ã3 = Q3.T、残り R3.T を G2 の右ボンドへ吸収する。
Q3, R3 = torch.linalg.qr(A3.T, mode="reduced")
G3_right = Q3.T.unsqueeze(-1)  # (r2, n3) → (r2, n3, 1)
G2_right = torch.tensordot(G2, R3.T, dims=([2], [0]))

print("Q3 shape:", tuple(Q3.shape))
print("R3 shape:", tuple(R3.shape))
print("G2_right shape:", tuple(G2_right.shape))
print("G3_right shape:", tuple(G3_right.shape))


SyntaxError: unmatched ')' (4273103653.py, line 8)

## 5. 演習3 — 右直交性を確認する

右直交化後の第3コアを行列として見たものを

$$
\widetilde A_3\in\mathbb{R}^{r_2\times n_3}
$$

とします。

確認したいのは、

$$
\boxed{
\widetilde A_3\widetilde A_3^T
=
I_{r_2}
}
$$

です。

したがって、

$$
\left\|
\widetilde A_3\widetilde A_3^T-I_{r_2}
\right\|_F
$$

を計算してください。

float64 なら、正しく実装できていれば丸め誤差程度になります。

In [4]:
# TODO 3:
# 右直交性を確認してください。
#
# Ã3 = Q3.T ∈ R^{r2 × n3} について Ã3 Ã3^T = I_{r2} を確認する。
A3_right = G3_right.squeeze(-1)  # = Q3.T, shape (r2, n3)
I_r2 = torch.eye(A3_right.shape[0], dtype=A3_right.dtype, device=A3_right.device)
gram = A3_right @ A3_right.T
right_orthogonality_error = torch.linalg.matrix_norm(gram - I_r2, ord="fro").item()

print("A3_right @ A3_right.T =")
print(gram)
print("right orthogonality error =", right_orthogonality_error)


A3_right @ A3_right.T =
tensor([[ 1.0000e+00,  3.4694e-17, -7.6328e-17],
        [ 3.4694e-17,  1.0000e+00, -2.7756e-17],
        [-7.6328e-17, -2.7756e-17,  1.0000e+00]])
right orthogonality error = 4.0436549343196473e-16


## 6. 演習4 — $R^T$ の吸収で局所収縮が保存されるか

右直交化の前後で、第2–第3コア部分だけを比較します。

変換前：

$$
B_{\mathrm{before}}(\alpha_1,i_2,i_3)
=
\sum_{\alpha_2}
G_2(\alpha_1,i_2,\alpha_2)
G_3(\alpha_2,i_3,1)
$$

変換後：

$$
B_{\mathrm{after}}(\alpha_1,i_2,i_3)
=
\sum_{\beta_2}
\widetilde G_2(\alpha_1,i_2,\beta_2)
\widetilde G_3(\beta_2,i_3,1)
$$

です。

確認したいのは、

$$
\boxed{
\|B_{\mathrm{after}}-B_{\mathrm{before}}\|_F
\approx 0
}
$$

です。

これは、

> $R^T$ を第2コアへ正しく吸収したため、第2–第3コアの局所的な写像が変わっていない

ことを直接確認する検証です。

余裕があれば、最後に全テンソルも再構成して、

$$
X_{\mathrm{before}}\simeq X_{\mathrm{after}}
$$

も確認してください。

In [5]:
# TODO 4:
# 第2-第3コアの局所収縮保存を確認してください。
#
B_before = torch.tensordot(G2,G3, dims=([2], [0]))
B_after = torch.tensordot(G2_right,G3_right, dims=([2], [0]))
local_error = torch.norm(B_after-B_before)
#
# 余裕があれば全テンソルも比較
X_before = torch.tensordot(G1,B_before, dims=([2], [0]))
X_after = torch.tensordot(G1,B_after, dims=([2], [0]))

#下でもOK
#X_before = reconstruct_tt3(G1, G2, G3)
#X_after  = reconstruct_tt3(G1, G2_right, G3_right)

global_error = torch.norm(X_after-X_before)
#
print("local absorption error =",local_error)
print("global reconstruction error =", global_error)

print("TODO: R.T 吸収後も局所収縮が保存されることを確認する")


local absorption error = tensor(2.4389e-15)
global reconstruction error = tensor(6.2127e-15)
TODO: R.T 吸収後も局所収縮が保存されることを確認する


## 7. 追加演習 — 右部分収縮と右ブロック状態

ここでは新しい QR 操作は行いません。

すでに作った右直交化後の第3コア `G3_right` を使って、

- 右部分収縮 $R_2$ とは何か
- その shape はどうなるか
- $R_2$ の各行を「右ブロック状態」とどう解釈するか
- 右直交性が、右ブロック状態の正規直交性をどう意味するか

を自分でつなげます。

### 問1 — 右部分収縮 $R_2$ を定義する

3階 TT を第2–第3コア間の仮想ボンド $\alpha_2$ で切ります。

このとき右側に残るのは、右直交化済みの第3コアだけです。

次を自分で書いてください。

1. $R_2(\alpha_2,i_3)$ を `G3_right` を使って定義する
2. $R_2$ の shape を求める
3. すでに作った `A3_right` と $R_2$ がどのような関係にあるか説明する

### ヒント

`G3_right` の shape は

$$
(r_2,n_3,1)
$$

でした。

最後のサイズ $1$ の軸をどう扱えばよいか考えてください。

### 問2 — 右ブロック状態を定義する

仮想ボンド添字 $\alpha_2$ を固定したとき、$R_2$ の1行は第3サイト側の1つの状態ベクトルとみなせます。

次を自分で書いてください。

1. $\lvert R_{\alpha_2}\rangle$ を、物理基底 $\lvert i_3\rangle$ と $R_2(\alpha_2,i_3)$ を使って表す
2. 次の4つが何に対応するか説明する
   - $\lvert i_3\rangle$
   - $\alpha_2$
   - $R_2(\alpha_2,i_3)$
   - $R_2$ の第 $\alpha_2$ 行

---

### 問3 — 右直交性から状態の正規直交性を導く

すでに右直交化の演習で、

$$
R_2R_2^T \simeq I_{r_2}
$$

に対応する性質を確認しています。

今度はそれを「状態の内積」として読み替えます。

次の出発点から、

$$
\langle R_{\beta_2}\mid R_{\alpha_2}\rangle
$$

を自分で展開し、最終的に

$$
\delta_{\beta_2\alpha_2}
$$

が得られることを示してください。

途中で使う物理基底の性質も明示してください。

### ヒント

物理基底について、

$$
\langle j_3|i_3\rangle
$$

がどうなるかを使います。

また、最後に現れる和が行列積 $R_2R_2^T$ のどの成分に対応するか確認してください。

---

### 問4 — 左部分収縮との対応を整理する

前回扱った左部分収縮 $L_2$ と、今回の右部分収縮 $R_2$ を比較してください。

最低限、次の項目を表にしてください。

- shape
- 仮想ボンド添字 $\alpha_2$ が行側か列側か
- 直交性を表す Gram 行列
- 行または列が何の状態ベクトルに対応するか

最後に、

$$
L_2^TL_2
$$

と

$$
R_2R_2^T
$$

で転置の位置が違う理由を、**添字配置の違い**という観点から説明してください。

In [6]:
# TODO 5:
# 右部分収縮 R2 と、右ブロック状態の Gram 行列を自分で確認してください。
#
# 1. G3_right から R2 を作る
R2 = G3_right.squeeze(-1)
#
# 2. shape を確認する
print("R2 shape:", R2)
#
# 3. 右ブロック状態の Gram 行列を作る
right_block_gram = R2 @ R2.T       # (r2, r2)
#
# 4. 比較用の単位行列 I_{r2} を作る
I_r2 = torch.eye(right_block_gram.shape[1], dtype=right_block_gram.dtype, device=right_block_gram.device)
#
# 5. ||R2 R2^T - I||_F を計算する
right_block_error = torch.linalg.matrix_norm(right_block_gram - I_r2, ord="fro").item()
#
print(f"R2 @ R2.T =\n{right_block_gram}")
print("||R2 R2.T - I_r2||_F =", right_block_error )

print("TODO: 右部分収縮と右ブロック状態の正規直交性を確認する")


R2 shape: tensor([[-5.1665e-03, -6.2685e-01, -7.4618e-01, -1.8274e-01,  1.2984e-01],
        [ 1.3080e-01, -6.9232e-01,  6.1322e-01,  1.0888e-01,  3.4014e-01],
        [ 7.6965e-01,  1.0037e-05,  8.7517e-02, -5.7154e-01, -2.7078e-01]])
R2 @ R2.T =
tensor([[ 1.0000e+00,  3.4694e-17, -7.6328e-17],
        [ 3.4694e-17,  1.0000e+00, -2.7756e-17],
        [-7.6328e-17, -2.7756e-17,  1.0000e+00]])
||R2 R2.T - I_r2||_F = 4.0436549343196473e-16
TODO: 右部分収縮と右ブロック状態の正規直交性を確認する


### このNotebookの完了条件

この追加演習では、新しい変換は行いません。

自分で

- $R_2$ の定義と shape
- 右ブロック状態の意味
- 状態内積と $R_2R_2^T$ の対応
- 左部分収縮 $L_2$ との対称性
- 数値的な Gram 行列の確認

までつなげられれば、この右直交化Notebookは完了です。

## 8. 今回の到達点

今回の流れは、

$$
G_3
\rightarrow
A_3
\rightarrow
A_3^T=QR
\rightarrow
\widetilde G_3=Q^T
\rightarrow
R^T\text{ を }G_2\text{ へ吸収}
$$

です。

自分の実装で、

$$
\widetilde A_3 \widetilde A_3^T
\simeq
I_{r_2}
$$

と、

$$
B_{\mathrm{before}}
\simeq
B_{\mathrm{after}}
$$

を確認し、さらに右部分収縮

$$
R_2(\alpha_2, i_3)
=
\widetilde G_3(\alpha_2, i_3, 1)
$$

を定義して

$$
R_2 R_2^T
=
I_{r_2}
$$

が右ブロック状態の正規直交性を意味することを確認できれば完了です。

### 次に進むテーマ

次は新しい sweep を増やす前に、

- 右側ブロック状態とは何か
- 右直交コアをつないだとき、なぜ右ブロック全体も正規直交になるのか

を確認します。

そのあとで初めて、

$$
\text{mixed-canonical form}
$$

と

$$
\text{orthogonality center}
$$

へ進みます。